# 05c — SRM-teacher local detectability risk

The power-curve diagnostic showed that SRM-lite has substantial held-out detection power even at the frozen payload levels, whereas the old 8-feature auxiliary detector was at chance level.

This notebook therefore replaces the old block detector **for method development only** with an image-level SRM-lite teacher trained strictly on the training split.

For each block \(i\), a deterministic reversible probe is embedded only in that block and local risk is defined as

\[
D_i=\frac{\ell(X_i^{\mathrm{probe}})-\ell(X)}{n_i},
\]

where \(\ell\) is the SRM-teacher logit and \(n_i\) is the number of probe bits.

The normalization makes blocks with different probe capacities more comparable. Negative values are retained: a probe can make a particular image less stego-like for the teacher.

**Important:** SRM-lite becomes a *surrogate/teacher*, not the final independent detector. The residual CNN remains independent evidence later.


In [1]:
from pathlib import Path
import hashlib, json, joblib, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, kendalltau

from rdhlab.io import read_gray
from rdhlab.blocks import iter_blocks
from rdhlab.features import predictability_score
from rdhlab.detectors import fit_rich_detector, detector_metrics, paired_detector_bootstrap
from rdhlab.teacher_risk import SRMTeacherLocalRisk, rich_detector_logit
from rdhlab.pipeline import load_payload_freeze, run_frozen_image_precomputed, deterministic_seed
from rdhlab.blockcodec import analyze_blocks
from rdhlab.allocation import rank_blocks

config=yaml.safe_load(Path('/workspace/config/experiment.yaml').read_text())
seed=int(config['project']['seed'])
bs=int(config['dataset']['block_size'])
manifest=pd.read_csv(config['dataset']['prepared_manifest'])
train=manifest[manifest.split=='train'].reset_index(drop=True)
val=manifest[manifest.split=='validation'].reset_index(drop=True)
frozen=sorted(map(float,load_payload_freeze('/workspace/config/frozen_payloads.json')['levels']))
power=pd.read_csv('/workspace/results/detectability_power/srm_lite_power_curve.csv')
out=Path('/workspace/results/srm_teacher_risk'); out.mkdir(parents=True,exist_ok=True)
models=Path('/workspace/results/models'); models.mkdir(parents=True,exist_ok=True)

# Reproducible validation-only rule: smallest FROZEN payload with AUC >= .80
# and validation feasibility >= .95. Fall back to the best frozen AUC.
elig=power[(power.is_frozen==True) & (power.auc>=0.80) & (power.val_feasible_fraction>=0.95)].sort_values('payload_bpp')
if len(elig):
    teacher_bpp=float(elig.iloc[0].payload_bpp)
    selection_rule='smallest frozen payload with AUC>=0.80 and validation feasibility>=0.95'
else:
    q=power[power.is_frozen==True].sort_values('auc',ascending=False).iloc[0]
    teacher_bpp=float(q.payload_bpp)
    selection_rule='fallback: frozen payload with highest validation AUC'
print('Teacher payload:',teacher_bpp)
print('Selection rule:',selection_rule)


Teacher payload: 0.009
Selection rule: smallest frozen payload with AUC>=0.80 and validation feasibility>=0.95


## 1. Train the SRM-lite teacher on the training split only

A neutral random allocator is used to generate training stego images. No validation or test labels are used to fit the teacher.


In [2]:
class NeutralRiskModel:
    def __init__(self, block_size): self.block_size=int(block_size)
    def score_image_blocks(self,image,image_key=''):
        rows=[]
        for bid,y,x,b in iter_blocks(image,self.block_size):
            rows.append({'block_id':int(bid),'y':int(y),'x':int(x),
                         'predictability':float(predictability_score(b)),
                         'detectability_risk':0.0})
        return rows

neutral=NeutralRiskModel(bs)

# Helper independent of the old weak risk model.
from rdhlab.pipeline import run_frozen_image

def make_random_pairs(frame,n,bpp,label):
    C=[]; S=[]; ids=[]
    for j,row in frame.head(min(n,len(frame))).iterrows():
        x=read_gray(row.path)
        rr=run_frozen_image(x,str(row.source_id),bpp,'random',neutral,.5,bs,seed)
        if rr['feasible']:
            C.append(x); S.append(rr['stego']); ids.append(str(row.source_id))
        if (j+1)%250==0: print(label,j+1,'feasible',len(C))
    return C,S,ids

N_TEACHER_TRAIN=min(2000,len(train))
tr_c,tr_s,tr_ids=make_random_pairs(train,N_TEACHER_TRAIN,teacher_bpp,'teacher train')
print('Teacher training pairs:',len(tr_c))
teacher=fit_rich_detector(tr_c,tr_s,seed=seed)
joblib.dump(teacher,models/'srm_teacher.joblib')


teacher train 250 feasible 250
teacher train 500 feasible 497
teacher train 750 feasible 744
teacher train 1000 feasible 992
teacher train 1250 feasible 1233
teacher train 1500 feasible 1480
teacher train 1750 feasible 1728
teacher train 2000 feasible 1967
Teacher training pairs: 1967


['/workspace/results/models/srm_teacher.joblib']

## 2. Held-out confirmation of the newly fitted teacher

This is a sanity check on validation images, not test-set evaluation.


In [ ]:
N_TEACHER_VAL=min(1000,len(val))
va_c,va_s,va_ids=make_random_pairs(val,N_TEACHER_VAL,teacher_bpp,'teacher validation')
c=np.asarray([teacher.score(x) for x in va_c],float)
s=np.asarray([teacher.score(x) for x in va_s],float)
y=np.tile([0,1],len(c)); scores=np.column_stack([c,s]).reshape(-1)
m=detector_metrics(y,scores,fixed_fpr=float(config['detectors']['fixed_fpr']))
ci=paired_detector_bootstrap(c,s,fixed_fpr=float(config['detectors']['fixed_fpr']),
    n_resamples=int(config['statistics']['cluster_bootstrap_resamples']),
    confidence=float(config['statistics']['confidence']),seed=seed)
teacher_validation={**m,**ci,'payload_bpp':teacher_bpp,'n_pairs':len(c)}
print(teacher_validation)
(out/'teacher_validation.json').write_text(json.dumps(teacher_validation,indent=2),encoding='utf-8')


teacher validation 250 feasible 248
teacher validation 500 feasible 493
teacher validation 750 feasible 733
teacher validation 1000 feasible 967


## 3. Build the marginal local-risk model

The probe is deterministic for `(source_id, block_id)`. The teacher logit of the unchanged cover is computed once per image, then each block is probed independently.


In [ ]:
local_risk=SRMTeacherLocalRisk(
    teacher=teacher,
    block_size=bs,
    probe_fraction=float(config['risk_model']['probe_fraction']),
    probe_max_bits=128,
    seed=seed,
    normalize_by_bits=True,
)
joblib.dump(local_risk,models/'srm_teacher_local_risk.joblib')
print(local_risk)


## 4. Re-test the predictability–risk relation

This replaces the earlier relation obtained from the weak auxiliary detector. The experimental unit remains the source image; the scatter itself is descriptive.


In [ ]:
N_BLOCK_DIAG=min(1000,len(val))
rows=[]
for j,row in val.head(N_BLOCK_DIAG).iterrows():
    x=read_gray(row.path)
    br=local_risk.score_image_blocks(x,str(row.source_id))
    for r in br:
        rows.append({'source_id':str(row.source_id),**r})
    if (j+1)%100==0: print('risk blocks',j+1,'/',N_BLOCK_DIAG)
blocks=pd.DataFrame(rows)
blocks.to_csv(out/'validation_block_risk.csv',index=False)
rho,p_rho=spearmanr(blocks.predictability,blocks.detectability_risk)
tau,p_tau=kendalltau(blocks.predictability,blocks.detectability_risk)
print({'spearman_rho':rho,'spearman_p':p_rho,'kendall_tau':tau,'kendall_p':p_tau,'n_blocks':len(blocks)})


## 5. Validation-only alpha diagnostics with the new risk

No alpha is frozen here. We first need to verify that the new local risk actually changes image-level teacher detectability while retaining the expected PSNR trade-off.


In [ ]:
alpha_grid=list(map(float,config['allocator']['alpha_grid']))
N_ALPHA=min(750,len(val))

def make_orders(rows,source_id,alpha):
    bids=np.asarray([r['block_id'] for r in rows],int)
    p=np.asarray([r['predictability'] for r in rows],float)
    d=np.asarray([r['detectability_risk'] for r in rows],float)
    digest=hashlib.sha256(f'{seed}|{source_id}'.encode()).digest()
    rng=np.random.default_rng(int.from_bytes(digest[:8],'little'))
    rnd=bids.copy(); rng.shuffle(rnd)
    return {
        'raster':bids.copy(),
        'random':rnd,
        'predictability':bids[np.argsort(-p,kind='stable')],
        'detectability':bids[np.argsort(d,kind='stable')],
        'joint':bids[rank_blocks(p,d,alpha,1.0-alpha)],
    }

alpha_rows=[]
for j,row in val.head(N_ALPHA).iterrows():
    sid=str(row.source_id); x=read_gray(row.path)
    br=local_risk.score_image_blocks(x,sid)
    plans=analyze_blocks(x,bs)
    cover_logit=rich_detector_logit(teacher,x)
    for alpha in alpha_grid:
        orders=make_orders(br,sid,alpha)
        rr=run_frozen_image_precomputed(x,sid,teacher_bpp,'joint',orders,br,bs,seed,False,None,plans=plans)
        if not rr['feasible']:
            alpha_rows.append({'source_id':sid,'alpha':alpha,'feasible':False})
            continue
        stego_logit=rich_detector_logit(teacher,rr['stego'])
        alpha_rows.append({
            'source_id':sid,'alpha':alpha,'feasible':True,
            'teacher_cover_logit':cover_logit,'teacher_stego_logit':stego_logit,
            'teacher_delta_logit':stego_logit-cover_logit,
            'psnr':rr['psnr'],'ssim':rr['ssim'],
            'selected_P_mean':rr['selected_predictability_mean'],
            'selected_D_mean':rr['selected_detectability_risk_mean'],
            'used_blocks':rr['used_blocks'],
        })
    if (j+1)%100==0: print('alpha diag',j+1,'/',N_ALPHA)

ad=pd.DataFrame(alpha_rows)
ad.to_csv(out/'alpha_validation_pairs.csv',index=False)
summary=(ad[ad.feasible==True].groupby('alpha',as_index=False).agg(
    n=('source_id','size'),
    teacher_delta_mean=('teacher_delta_logit','mean'),
    teacher_delta_median=('teacher_delta_logit','median'),
    psnr_mean=('psnr','mean'),
    ssim_mean=('ssim','mean'),
    selected_P_mean=('selected_P_mean','mean'),
    selected_D_mean=('selected_D_mean','mean'),
    used_blocks_mean=('used_blocks','mean'),
))
summary['feasible_fraction']=ad.groupby('alpha').feasible.mean().values
summary.to_csv(out/'alpha_validation_summary.csv',index=False)
display(summary)


In [ ]:
fig,ax=plt.subplots(figsize=(6.2,4.2))
ax.plot(summary.alpha,summary.teacher_delta_median,marker='o')
ax.axhline(0,linewidth=1)
ax.set_xlabel(r'$\alpha$')
ax.set_ylabel('Median SRM-teacher logit change')
ax.set_title('Validation detectability surrogate versus allocator weight')
ax.grid(True,alpha=.2); fig.tight_layout()
fig.savefig(out/'alpha_vs_teacher_delta.png',dpi=300); plt.show()

fig,ax=plt.subplots(figsize=(6.2,4.2))
ax.plot(summary.alpha,summary.psnr_mean,marker='o')
ax.set_xlabel(r'$\alpha$'); ax.set_ylabel('Mean PSNR (dB)')
ax.set_title('Validation distortion versus allocator weight')
ax.grid(True,alpha=.2); fig.tight_layout()
fig.savefig(out/'alpha_vs_psnr.png',dpi=300); plt.show()


## Decision checkpoint

Do **not** overwrite `frozen_allocator.json` yet.

The new risk model is promising only if:

1. the fitted teacher remains clearly above chance on held-out validation;
2. lower selected \(D_i\) produces a measurable reduction in teacher logit change;
3. the PSNR trade-off remains non-trivial rather than collapsing all alpha values to the same blocks.

After this checkpoint, the next step is an **independent residual-CNN transfer test**. Only if the improvement transfers to the CNN should the allocator be frozen for the 2000-image test experiment.
